# 04 — Pemodelan: Random Forest + TF-IDF (Default vs Tuning)

Melatih **Random Forest** untuk klasifikasi sentimen biner (0=negatif, 1=positif)
dengan ekstraksi fitur **TF-IDF**.

- **TF-IDF di-*fit* hanya pada data latih** lalu dipakai untuk data uji (mencegah *data leakage*).
- **Model A (default):** `RandomForestClassifier` dengan parameter bawaan.
- **Model B (tuning):** `GridSearchCV` (5-fold, `scoring='f1'`).
- Perbandingan diuji pada data uji yang sama untuk menjawab **RQ3**.

Input : `data/processed/train.csv`, `data/processed/test.csv`  
Output: `model/tfidf_vectorizer.joblib`, `model/random_forest_default.joblib`,
`model/random_forest_tuned.joblib`, `data/processed/prediksi_test.csv`,
`model/perbandingan_default_vs_tuned.csv`

In [1]:
from pathlib import Path
import time

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV

ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA_PROCESSED = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_PROCESSED / "train.csv")
test = pd.read_csv(DATA_PROCESSED / "test.csv")
print("Latih:", train.shape, "| Uji:", test.shape)

X_train_txt, y_train = train["text_bersih"].fillna(""), train["label"]
X_test_txt, y_test = test["text_bersih"].fillna(""), test["label"]

Latih: (3614, 6) | Uji: (904, 6)


In [2]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
X_train = tfidf.fit_transform(X_train_txt)
X_test = tfidf.transform(X_test_txt)
print("Dimensi fitur TF-IDF (latih):", X_train.shape)
print("Dimensi fitur TF-IDF (uji)  :", X_test.shape)
print("Contoh fitur:", tfidf.get_feature_names_out()[:15])

Dimensi fitur TF-IDF (latih): (3614, 4529)
Dimensi fitur TF-IDF (uji)  : (904, 4529)
Contoh fitur: ['aamiin' 'abad' 'abis' 'abis baru' 'abis padahal' 'abis semua'
 'abis terus' 'acces' 'acces jangan' 'acces kai' 'access' 'access kai'
 'access lambat' 'access sangat' 'access tidak']


In [3]:
def evaluasi(nama, model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    return {
        "model": nama,
        "akurasi": accuracy_score(y, pred),
        "precision": precision_score(y, pred),
        "recall": recall_score(y, pred),
        "f1": f1_score(y, pred),
        "auc": roc_auc_score(y, proba),
    }

hasil = []

In [4]:
rf_default = RandomForestClassifier(random_state=42)
t0 = time.time()
rf_default.fit(X_train, y_train)
print(f"Waktu latih (default): {time.time() - t0:.1f} detik")

hasil.append(evaluasi("RF default (latih)", rf_default, X_train, y_train))
hasil.append(evaluasi("RF default (uji)", rf_default, X_test, y_test))
pd.DataFrame(hasil).round(4)

Waktu latih (default): 1.5 detik


,model,akurasi,precision,recall,f1,auc
0,RF default (latih),0.9956,0.9961,0.9961,0.9961,0.9986
1,RF default (uji),0.9082,0.9532,0.8802,0.9152,0.9545


In [5]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 20, 40],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1,
)
t0 = time.time()
grid.fit(X_train, y_train)
print(f"Waktu tuning: {time.time() - t0:.1f} detik")
print("Parameter terbaik:", grid.best_params_)
print("F1 rata-rata validasi silang terbaik:", round(grid.best_score_, 4))

Fitting 5 folds for each of 144 candidates, totalling 720 fits


Waktu tuning: 80.3 detik
Parameter terbaik: {'class_weight': None, 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}
F1 rata-rata validasi silang terbaik: 0.9047


In [6]:
rf_tuned = grid.best_estimator_

hasil.append(evaluasi("RF tuning (latih)", rf_tuned, X_train, y_train))
hasil.append(evaluasi("RF tuning (uji)", rf_tuned, X_test, y_test))

tabel = pd.DataFrame(hasil)
print(tabel.round(4).to_string(index=False))

tabel.to_csv(MODEL_DIR / "perbandingan_default_vs_tuned.csv", index=False)
print("\nTersimpan: model/perbandingan_default_vs_tuned.csv")

             model  akurasi  precision  recall     f1    auc
RF default (latih)   0.9956     0.9961  0.9961 0.9961 0.9986
  RF default (uji)   0.9082     0.9532  0.8802 0.9152 0.9545
 RF tuning (latih)   0.9947     0.9966  0.9941 0.9953 0.9983
   RF tuning (uji)   0.9049     0.9471  0.8802 0.9124 0.9567

Tersimpan: model/perbandingan_default_vs_tuned.csv


In [7]:
joblib.dump(tfidf, MODEL_DIR / "tfidf_vectorizer.joblib")
joblib.dump(rf_default, MODEL_DIR / "random_forest_default.joblib")
joblib.dump(rf_tuned, MODEL_DIR / "random_forest_tuned.joblib")

prediksi = test[["reviewId", "content", "text_bersih", "sentimen", "label"]].copy()
prediksi["pred_default"] = rf_default.predict(X_test)
prediksi["proba_default"] = rf_default.predict_proba(X_test)[:, 1]
prediksi["pred_tuned"] = rf_tuned.predict(X_test)
prediksi["proba_tuned"] = rf_tuned.predict_proba(X_test)[:, 1]
prediksi.to_csv(DATA_PROCESSED / "prediksi_test.csv", index=False, encoding="utf-8-sig")

for f in ["tfidf_vectorizer.joblib", "random_forest_default.joblib", "random_forest_tuned.joblib"]:
    print(f" - model/{f} ({(MODEL_DIR / f).stat().st_size} byte)")
print(f" - data/processed/prediksi_test.csv ({len(prediksi)} baris)")

 - model/tfidf_vectorizer.joblib (106675 byte)
 - model/random_forest_default.joblib (12236953 byte)
 - model/random_forest_tuned.joblib (45513913 byte)
 - data/processed/prediksi_test.csv (904 baris)


## Catatan

- TF-IDF di-*fit* pada data latih saja, lalu mentransformasi data uji.
- Model default vs hasil tuning dibandingkan pada data uji yang sama.
- Pencarian hyperparameter memakai 5-fold cross-validation dengan metrik F1-score.
- Evaluasi rinci (confusion matrix, kurva ROC, feature importance) ada di `05_evaluasi.ipynb`.